# Strategy Performance Analysis

This notebook evaluates the performance of the statistical arbitrage strategy.

Steps performed:

1. Load portfolio returns from multi-pair backtest
2. Compute portfolio equity curve
3. Compare strategy performance with NIFTY50 benchmark
4. Calculate rolling Sharpe ratio
5. Compute performance metrics (Return, Volatility, Sharpe, Drawdown)
6. Analyze profit contribution from trading pairs
7. Identify top performing pairs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf


In [ ]:
portfolio_returns = pd.read_csv("../data/processed/portfolio_returns.csv",
                                index_col=0,
                                parse_dates=True)

portfolio_returns = portfolio_returns.squeeze()


In [ ]:
cumulative_returns = (1 + portfolio_returns).cumprod()

plt.figure(figsize=(12,6))
plt.plot(cumulative_returns)

plt.title("Statistical Arbitrage Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.grid(True)

plt.savefig("../reports/equity_curve.png", dpi=300)
plt.show()


In [ ]:
nifty = yf.download("^NSEI",
                    start=cumulative_returns.index.min(),
                    end=cumulative_returns.index.max())

nifty_returns = nifty["Close"].pct_change().fillna(0)

nifty_curve = (1 + nifty_returns).cumprod()


In [ ]:
plt.figure(figsize=(12,6))

plt.plot(cumulative_returns, label="Stat Arb Strategy")
plt.plot(nifty_curve, label="NIFTY50 Benchmark")

plt.title("Strategy vs Market")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")

plt.legend()
plt.grid(True)
plt.savefig("../reports/strategy_vs_nifty.png", dpi=300)
plt.show()


In [ ]:
rolling_sharpe = (
    portfolio_returns.rolling(252).mean() /
    portfolio_returns.rolling(252).std()
)*np.sqrt(252)

plt.figure(figsize=(12,5))
plt.plot(rolling_sharpe)

plt.title("Rolling Sharpe Ratio")
plt.xlabel("Date")
plt.ylabel("Sharpe")

plt.grid(True)
plt.savefig("../reports/rolling_sharpe_ratio.png", dpi=300)
plt.show()


In [ ]:
total_return = cumulative_returns.iloc[-1] - 1

annual_return = portfolio_returns.mean() * 252
volatility = portfolio_returns.std() * np.sqrt(252)

risk_free_rate = 0.05 / 252

excess_returns = portfolio_returns - risk_free_rate
if excess_returns.std()!=0:

 sharpe = np.sqrt(252) * excess_returns.mean() / excess_returns.std()
else:
  sharpe=0

downside_returns = portfolio_returns[portfolio_returns < 0]
downside_std = downside_returns.std()

if downside_std != 0:
    sortino = np.sqrt(252) * portfolio_returns.mean() / downside_std
else:
    sortino = 0
drawdown = cumulative_returns / cumulative_returns.cummax() - 1
max_dd = drawdown.min()

calmar = annual_return / abs(max_dd)

metrics = pd.DataFrame({
    "Total Return":[total_return],
    "Annual Return":[annual_return],
    "Volatility":[volatility],
    "Sharpe":[sharpe],
    "Sortino":[sortino],
    "Max Drawdown":[max_dd],
    "Calmar":[calmar]
})

metrics


In [ ]:
plt.figure(figsize=(12,6))

plt.plot(drawdown)
plt.fill_between(drawdown.index, drawdown, 0)

plt.title("Portfolio Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")

plt.grid(True)
plt.savefig("../reports/portfolio_drawdown.png", dpi=300)
plt.show()

In [ ]:
metrics.to_csv("../data/processed/strategy_metrics.csv", index=False)

print("Strategy metrics saved successfully")

In [ ]:
pair_performance = pd.read_csv("../data/processed/pair_performance.csv",
                               index_col=0)


In [ ]:
pairs_list = pair_performance.index
profits = pair_performance["profit"]

plt.figure(figsize=(12,6))
sorted_pairs = pair_performance.sort_values("profit", ascending=False)

top_pairs_plot = sorted_pairs.head(30)

plt.figure(figsize=(12,6))

plt.bar(range(len(top_pairs_plot)), top_pairs_plot["profit"])

plt.title("Profit Contribution by Top 30 Pairs")
plt.xlabel("Pairs Rank")
plt.ylabel("Total Profit")

plt.grid(True)
plt.savefig("../reports/pair_profit_distribution.png", dpi=300)
plt.show()



In [ ]:
top_pairs = pair_performance.sort_values("profit", ascending=False)

print("Top 5 Pairs:")
print(top_pairs.head())


### Strategy Performance Summary

This notebook evaluates the performance of the statistical arbitrage pairs trading strategy.

**Key observations**

- The strategy generated strong cumulative returns.
- Rolling Sharpe ratio indicates stable risk-adjusted performance.
- Portfolio drawdowns remained within acceptable limits.
- Several cointegrated pairs produced consistent trading profits.

**Conclusion**

The results show that cointegration-based pairs trading can effectively exploit mean-reversion opportunities in the market.